# 検索用の埋め込みモデルを自分のデータで微調整する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_search/embedding_finetune.ipynb)

NLP クックブック 20-17 ／ 検索

汎用の埋め込みモデルでは、自分の分野の検索で精度が足りない。クエリと正解文書の組から、埋め込みモデルを追加学習したい。

In [1]:
!pip install -q datasets accelerate

In [2]:
import io, tarfile, urllib.request
import numpy as np
import pandas as pd

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

# (タイトル, 本文) を (クエリ, 正解文書) の組とみなす。本文の冒頭はタイトルと重なりやすいので、途中の部分を使って難しくする
df = load_livedoor().sample(2100, random_state=0).reset_index(drop=True)
df["passage"] = df["text"].map(lambda t: " ".join(t.split())[200:500])
df = df[df["passage"].str.len() > 100].reset_index(drop=True)
train, test = df.iloc[:1200], df.iloc[1200:1800].reset_index(drop=True)
print(len(train), "組で学習 /", len(test), "組で評価")

1200 組で学習 / 600 組で評価


In [3]:
from sentence_transformers import SentenceTransformer

def evaluate(model):
    q = model.encode(["query: " + t for t in test["title"]], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    d = model.encode(["passage: " + t for t in test["passage"]], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    ranking = np.argsort(-(q @ d.T), axis=1)
    rank = np.array([np.where(ranking[i] == i)[0][0] + 1 for i in range(len(test))])
    return {"Recall@1": round(float((rank <= 1).mean()), 3), "Recall@10": round(float((rank <= 10).mean()), 3), "MRR": round(float((1 / rank).mean()), 3)}

model = SentenceTransformer("intfloat/multilingual-e5-small")
model.max_seq_length = 128
print("微調整前:", evaluate(model))

微調整前: {'Recall@1': 0.715, 'Recall@10': 0.9, 'MRR': 0.782}


In [4]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

train_ds = Dataset.from_dict({"anchor": ["query: " + t for t in train["title"]], "positive": ["passage: " + t for t in train["passage"]]})
loss = MultipleNegativesRankingLoss(model)   # バッチ内の他の文書を「不正解」として使う対照学習
args = SentenceTransformerTrainingArguments(output_dir="e5-finetuned", num_train_epochs=1, per_device_train_batch_size=32, learning_rate=2e-5,
                                            warmup_steps=5, logging_steps=10, save_strategy="no", report_to="none", seed=0)
SentenceTransformerTrainer(model=model, args=args, train_dataset=train_ds, loss=loss).train()
print("微調整後:", evaluate(model))
model.save("e5-finetuned")

Step,Training Loss
10,1.954177
20,0.892551
30,0.482021


微調整後: {'Recall@1': 0.723, 'Recall@10': 0.905, 'MRR': 0.789}


- 学習データは (クエリ, 正解文書) の組だけでよい。バッチ内の他の組の文書が自動的に不正解の例になるので、バッチサイズが大きいほど学習が効く。
- 1200 組・1 エポック (CPU で数分) では、改善はごくわずか (Recall@1 で 0.715 → 0.723)。はっきりした効果を得るには、数万組以上のデータと GPU での学習が要る。学習データには、実際のクエリのログや、LLM に文書から作らせた質問 (質問応答の章の質問生成) が学習データになる。
- 「正解ではないが紛らわしい文書」(ハードネガティブ) を BM25 や元のモデルの検索結果から集めて加えると、さらに伸びる。
- 評価データは必ず学習に使っていない組にする。微調整後は、分野外のクエリでの性能が下がっていないかも確かめる。